# Reproducibility notebook: hidden CKD review prioritization with PULSCAR and XGBoost

**Journal-facing computational supplement — version 1.0**

This notebook reconstructs and audits the locked primary analysis reported in the manuscript. It contains **no patient-level data, direct identifiers, private linkage files, or clinician responses**.

The analysis treats documented CKD records as observed positives and all other records as unlabeled. Official PULSCAR estimates the hidden-positive fraction within the training unlabeled pool; probable positives are then incorporated into a mean-imputed XGBoost ranking model. The continuous output is a **review-priority score**, not a calibrated CKD probability or diagnosis.

Two execution modes are supported:

1. `specification_check` (default): runs without restricted EHR data and verifies the frozen design, expected results, formulas, and software interface.
2. `authorized_full_rerun`: requires the deidentified frozen cohort files and audited PULSCAR software bundle described below.

**Primary scope:** cohort contracts, feature policy, training-only PULSCAR/model fitting, outcome-blind validation/test scoring, locked evaluation, bootstrap confidence intervals, eGFR comparators, and global SHAP. The exploratory clinician-review study is intentionally separated from primary model reproducibility.

## 1. Required restricted inputs and directory layout

The hospital EHR cannot be redistributed with the article. Authorized investigators should place the following deidentified, frozen files under `data/`:

```text
data/
  training.parquet                 # predictors + observed label + locked fold + temporal weight
  validation_predictors.parquet    # predictors only
  validation_outcomes.parquet      # patient key + observed label
  internal_test_predictors.parquet # predictors only
  internal_test_outcomes.parquet   # patient key + observed label
  pulscar_software_lock.json        # official implementation, archived wheel hash, parameters
  pulscar_official.whl              # archived audited wheel (authorized rerun only)
```

Patient identifiers must remain pseudonymized and must never be committed to a public repository. Validation and test outcomes are stored separately so that scores can be frozen before outcome linkage.

In [ ]:
# Configuration — edit only these paths/mode.
from pathlib import Path
import os

# Leave this as None for the public-safe specification check. For an
# authorized rerun in Colab, change only this value to
# "authorized_full_rerun" after the frozen deidentified inputs are ready.
EXECUTION_MODE_OVERRIDE = None
DATA_DIR_OVERRIDE = None       # Example: "/content/drive/MyDrive/CKD_repro/data"
OUTPUT_DIR_OVERRIDE = None     # Example: "/content/drive/MyDrive/CKD_repro/reproduced_outputs"
EXECUTION_MODE = EXECUTION_MODE_OVERRIDE or os.getenv("CKD_REPRO_MODE", "specification_check")
if EXECUTION_MODE not in {"specification_check", "authorized_full_rerun"}:
    raise ValueError("CKD_REPRO_MODE must be specification_check or authorized_full_rerun")

NOTEBOOK_DIR = Path.cwd()
DATA_DIR = Path(DATA_DIR_OVERRIDE or os.getenv("CKD_REPRO_DATA", NOTEBOOK_DIR / "data"))
OUTPUT_DIR = Path(OUTPUT_DIR_OVERRIDE or os.getenv("CKD_REPRO_OUTPUT", NOTEBOOK_DIR / "reproduced_outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_PULSCAR_DISCOVERY = EXECUTION_MODE == "authorized_full_rerun"
RUN_SHAP = EXECUTION_MODE == "authorized_full_rerun"
BOOTSTRAP_REPLICATES = 500
VALIDATION_BOOTSTRAP_SEED = 51_290
TEST_BOOTSTRAP_SEED = 51_212
EGFR_COMPARATOR_BOOTSTRAP_SEED = 51_213
MODEL_SEED = 71_090_046
PULSCAR_SEED = 60_123

print({"execution_mode": EXECUTION_MODE, "data_dir": str(DATA_DIR), "output_dir": str(OUTPUT_DIR)})

## 2. Frozen analysis specification

All feature decisions and model development precede outcome linkage in held-out cohorts. Features with training missingness strictly greater than 70% were excluded. Numeric features use training-mean imputation; gender is one-hot encoded. No missingness indicators, native-NaN routing, validation recalibration, or test-set threshold optimization are used.

In [ ]:
import hashlib, importlib.metadata, json, math, platform, sys, time
import numpy as np
import pandas as pd

RETAINED_FEATURES = [
    "AGE_AT_INDEX", "GENDER", "BMI_MODEL", "SBP", "DBP", "PULSE_FINAL",
    "DERIVED_EGFR_CKD_EPI_2021", "BUN_FINAL_MMOL_L",
    "POTASSIUM_FINAL_MMOL_L", "SODIUM_FINAL_MMOL_L",
    "HEMOGLOBIN_MODEL_G_L", "MCV_MODEL_FL", "RDW_CV_MODEL_PERCENT",
    "WBC_MODEL_10E9_L", "PLATELETS_MODEL_10E9_L", "MPV_MODEL_FL",
    "LYMPHOCYTES_ABSOLUTE_MODEL_10E9_L",
    "EOSINOPHILS_ABSOLUTE_MODEL_10E9_L", "EGFR_MIN_24M",
    "EGFR_MEAN_24M",
]
EXCLUDED_FEATURES = [
    "UACR_LOG1P_FINAL",
    "EGFR_PREVIOUS_90D",
    "EGFR_CHANGE_LAST_MINUS_PREVIOUS_90D",
    "EGFR_DAYS_BETWEEN_90D_MEASUREMENTS",
    "EGFR_SLOPE_PER_30_DAYS_90D",
]

COHORT_CONTRACT = pd.DataFrame([
    {"cohort": "Training", "patients": 69_701, "documented_ckd": 5_042, "unlabeled": 64_659},
    {"cohort": "Validation", "patients": 14_923, "documented_ckd": 1_089, "unlabeled": 13_834},
    {"cohort": "Locked internal test", "patients": 14_927, "documented_ckd": 1_071, "unlabeled": 13_856},
])

LOCKED_XGB_PARAMETERS = {
    "n_estimators": 700,
    "learning_rate": 0.04590659413126398,
    "reg_lambda": 13.008281483228425,
    "colsample_bytree": 0.7953856488673938,
    "max_depth": 5,
    "min_child_weight": 5.598456360046426,
    "subsample": 0.6746041940331612,
}
EXPECTED_ALPHA = 0.0041
EXPECTED_TRAIN_SELECTED = 265

assert len(RETAINED_FEATURES) == 20 and RETAINED_FEATURES.count("GENDER") == 1
assert len(EXCLUDED_FEATURES) == 5 and "UACR_LOG1P_FINAL" in EXCLUDED_FEATURES
display(COHORT_CONTRACT)
display(pd.DataFrame({"retained_feature": RETAINED_FEATURES}))

## 3. Reproducibility utilities and software record

File hashes protect the frozen input and output state. The software table generated here should be archived with every rerun. Exact production versions and the official PULSCAR wheel hash must also be retained in the run manifest; this notebook does not silently substitute another PULSCAR implementation.

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

def patient_key(series):
    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.notna().sum() == series.notna().sum():
        nonmissing = numeric.dropna()
        if (np.abs(nonmissing - np.round(nonmissing)) < 1e-9).all():
            return numeric.round().astype("Int64").astype("string")
    return series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)

def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"

software = pd.DataFrame([
    {"component": "python", "version": platform.python_version()},
    *[{"component": p, "version": package_version(p)} for p in
      ["numpy", "pandas", "pyarrow", "scikit-learn", "xgboost", "catboost", "optuna", "joblib", "shap", "matplotlib"]],
])
software.to_csv(OUTPUT_DIR / "software_versions.csv", index=False)
display(software)

if EXECUTION_MODE == "authorized_full_rerun":
    from packaging.specifiers import SpecifierSet
    required_versions = {
        "numpy": ">=2,<3",
        "pandas": ">=2.2,<4",
        "pyarrow": ">=18,<26",
        "scikit-learn": ">=1.9,<1.10",
        "xgboost": ">=3,<4",
        "catboost": "==1.2.10",
        "optuna": ">=4,<5",
        "joblib": ">=1.4,<2",
        "shap": ">=0.48,<1",
        "matplotlib": ">=3.9,<4",
    }
    incompatible = []
    for component, requirement in required_versions.items():
        installed = package_version(component)
        if installed == "not installed" or installed not in SpecifierSet(requirement):
            incompatible.append(f"{component} {installed!r} does not satisfy {requirement}")
    if incompatible:
        raise RuntimeError(
            "Authorized rerun environment is incompatible. Install the required versions, "
            "restart the Colab session, and rerun from the top:\n" + "\n".join(incompatible)
        )

## 4. Load and audit frozen cohorts

In the authorized mode, this cell enforces exact cohort sizes, unique keys, binary observed labels, predictor/outcome separation, and the expected feature schema. Column names for the pseudonymous patient key, observed label, locked fold, and temporal weight can be changed in the four constants below if the data custodian used equivalent names.

In [ ]:
ID_COL = "patient_id"
LABEL_COL = "observed_label"
FOLD_COL = "locked_fold"
WEIGHT_COL = "TEMPORAL_BALANCING_WEIGHT"

paths = {
    "training": DATA_DIR / "training.parquet",
    "validation_predictors": DATA_DIR / "validation_predictors.parquet",
    "validation_outcomes": DATA_DIR / "validation_outcomes.parquet",
    "test_predictors": DATA_DIR / "internal_test_predictors.parquet",
    "test_outcomes": DATA_DIR / "internal_test_outcomes.parquet",
    "software_lock": DATA_DIR / "pulscar_software_lock.json",
    "pulscar_wheel": DATA_DIR / "pulscar_official.whl",
}

EXPECTED_COHORTS = {
    "training": {"patients": 69_701, "documented_ckd": 5_042},
    "validation": {"patients": 14_923, "documented_ckd": 1_089},
    "test": {"patients": 14_927, "documented_ckd": 1_071},
}

def normalized_keys(frame):
    keys = patient_key(frame[ID_COL])
    if keys.isna().any() or keys.duplicated().any():
        raise ValueError("Patient keys are missing or duplicated")
    return keys

def audit_predictor_frame(frame, expected_n, training_frame=False):
    required = [ID_COL, *RETAINED_FEATURES]
    if training_frame:
        required += [LABEL_COL, FOLD_COL, WEIGHT_COL, *EXCLUDED_FEATURES]
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    if len(frame) != expected_n:
        raise ValueError(f"Expected {expected_n:,} rows; found {len(frame):,}")
    keys = normalized_keys(frame)
    if not training_frame and LABEL_COL in frame.columns:
        raise ValueError("Outcome label was found in an outcome-blind predictor file")
    if training_frame:
        labels = pd.to_numeric(frame[LABEL_COL], errors="raise")
        if not labels.isin([0, 1]).all() or int(labels.sum()) != EXPECTED_COHORTS["training"]["documented_ckd"]:
            raise ValueError("Training-label contract changed")
        folds = pd.to_numeric(frame[FOLD_COL], errors="raise")
        if folds.isna().any() or folds.nunique() != 5:
            raise ValueError("Expected exactly five nonmissing locked training folds")
        weights = pd.to_numeric(frame[WEIGHT_COL], errors="raise").to_numpy(float)
        if not np.isfinite(weights).all() or np.any(weights <= 0):
            raise ValueError("Temporal balancing weights must be finite and positive")
    return keys

def audit_outcomes(outcomes, predictors, cohort_name):
    contract = EXPECTED_COHORTS[cohort_name]
    missing = sorted({ID_COL, LABEL_COL} - set(outcomes.columns))
    if missing:
        raise ValueError(f"Missing outcome columns: {missing}")
    if len(outcomes) != contract["patients"]:
        raise ValueError(f"{cohort_name} outcome row count changed")
    outcome_keys = normalized_keys(outcomes)
    predictor_keys = normalized_keys(predictors)
    if set(outcome_keys) != set(predictor_keys):
        raise ValueError(f"{cohort_name} predictor/outcome patient keys do not match exactly")
    labels = pd.to_numeric(outcomes[LABEL_COL], errors="raise")
    if not labels.isin([0, 1]).all() or int(labels.sum()) != contract["documented_ckd"]:
        raise ValueError(f"{cohort_name} outcome-label contract changed")

if EXECUTION_MODE == "authorized_full_rerun":
    missing_files = [str(p) for p in paths.values() if not p.is_file()]
    if missing_files:
        raise FileNotFoundError("Required restricted inputs are missing:\n" + "\n".join(missing_files))
    training = pd.read_parquet(paths["training"])
    validation_x = pd.read_parquet(paths["validation_predictors"])
    validation_y = pd.read_parquet(paths["validation_outcomes"])
    test_x = pd.read_parquet(paths["test_predictors"])
    test_y = pd.read_parquet(paths["test_outcomes"])
    audit_predictor_frame(training, 69_701, training_frame=True)
    audit_predictor_frame(validation_x, 14_923)
    audit_predictor_frame(test_x, 14_927)
    audit_outcomes(validation_y, validation_x, "validation")
    audit_outcomes(test_y, test_x, "test")
    print("Restricted cohort contracts passed.")
else:
    training = validation_x = validation_y = test_x = test_y = None
    print("Specification check: restricted data were not loaded.")

## 5. Reproduce the training-only feature policy

The rule is data-derived but fixed before validation/test access: exclude a candidate feature only when its missingness in the frozen training cohort is **strictly greater than 70%**. The asserted exclusion set prevents accidental drift.

In [ ]:
CANDIDATE_FEATURES = RETAINED_FEATURES + EXCLUDED_FEATURES

if training is not None:
    availability = pd.DataFrame({
        "feature": CANDIDATE_FEATURES,
        "missing_percent": [100.0 * training[f].isna().mean() for f in CANDIDATE_FEATURES],
    })
    availability["policy"] = np.where(availability["missing_percent"] > 70.0, "exclude", "retain")
    reproduced_excluded = availability.loc[availability.policy.eq("exclude"), "feature"].tolist()
    reproduced_retained = [f for f in CANDIDATE_FEATURES if f not in reproduced_excluded]
    if set(reproduced_excluded) != set(EXCLUDED_FEATURES):
        raise AssertionError(f"Feature policy drift: {reproduced_excluded}")
    if reproduced_retained != RETAINED_FEATURES:
        raise AssertionError("Retained feature order changed")
    availability.to_csv(OUTPUT_DIR / "training_feature_policy.csv", index=False)
    display(availability)
else:
    print("Expected exclusion set:", EXCLUDED_FEATURES)

## 6. Leakage-safe preprocessing

Mean imputation and one-hot encoding are fitted on training data only. For nested development they must be fitted separately inside every training fold; for the final locked refit they are fitted once on the full training cohort. The transformation produces 19 numeric columns and two gender indicators (21 columns total).

In [ ]:
if EXECUTION_MODE == "authorized_full_rerun":
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC_FEATURES = [f for f in RETAINED_FEATURES if f != "GENDER"]

def fit_preprocessor(frame):
    numeric = frame[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    gender = frame[["GENDER"]].astype("string")
    imputer = SimpleImputer(strategy="mean", keep_empty_features=True)
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
    x_num = imputer.fit_transform(numeric).astype(np.float32)
    x_gender = encoder.fit_transform(gender).astype(np.float32)
    names = NUMERIC_FEATURES + encoder.get_feature_names_out(["GENDER"]).astype(str).tolist()
    matrix = np.column_stack([x_num, x_gender]).astype(np.float32)
    if matrix.shape[1] != 21 or not np.isfinite(matrix).all():
        raise ValueError("Unexpected transformed training matrix")
    return matrix, {"imputer": imputer, "encoder": encoder, "names": names}

def transform(frame, preprocessor):
    numeric = frame[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    gender = frame[["GENDER"]].astype("string")
    matrix = np.column_stack([
        preprocessor["imputer"].transform(numeric),
        preprocessor["encoder"].transform(gender),
    ]).astype(np.float32)
    if matrix.shape[1] != 21 or not np.isfinite(matrix).all():
        raise ValueError("Unexpected transformed evaluation matrix")
    return matrix

if training is not None:
    x_train, preprocessor = fit_preprocessor(training)
    print(x_train.shape, preprocessor["names"][-2:])

## 7. Official PULSCAR discovery on training only

This cell mirrors the locked discovery logic. It installs only the archived, hash-checked official wheel named in `pulscar_software_lock.json`, builds a separately standardized discovery matrix, and ranks unlabeled records by the official calibrated score. The number selected uses half-up rounding: `floor(alpha × N_unlabeled + 0.5)`. No validation or test outcomes are involved.

The software-lock JSON must contain `official_implementation.archived_wheel_sha256` and `shared_PULSNAR_parameters`. The latter is used verbatim with `scar=True`.

In [ ]:
def run_official_pulscar(training, lock_path, wheel_path):
    import contextlib, io, shutil, subprocess, tempfile

    lock = json.loads(Path(lock_path).read_text(encoding="utf-8"))
    expected_wheel_hash = lock["official_implementation"]["archived_wheel_sha256"]
    if sha256_file(wheel_path) != expected_wheel_hash:
        raise ValueError("The archived official PULSCAR wheel hash changed")
    original_wheel_name = Path(lock["official_implementation"]["archived_wheel"]).name
    if not original_wheel_name.endswith(".whl") or original_wheel_name == "pulscar_official.whl":
        raise ValueError("The software lock does not contain the original valid wheel filename")
    with tempfile.TemporaryDirectory(prefix="pulscar_wheel_install_") as install_dir:
        install_wheel = Path(install_dir) / original_wheel_name
        shutil.copy2(wheel_path, install_wheel)
        if sha256_file(install_wheel) != expected_wheel_hash:
            raise ValueError("The temporary install-wheel copy changed")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "--quiet", "--no-deps",
            "--force-reinstall", str(install_wheel),
        ])
    from PULSNAR import PULSNAR

    numeric = training[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    gender = training[["GENDER"]].astype("string")
    d_imputer = SimpleImputer(strategy="mean", keep_empty_features=True)
    d_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float64)
    discovery_x = np.column_stack([
        d_imputer.fit_transform(numeric),
        d_encoder.fit_transform(gender),
    ])
    discovery_x = StandardScaler().fit_transform(discovery_x).astype(np.float64)
    labels = pd.to_numeric(training[LABEL_COL], errors="raise").astype(int).to_numpy()
    shared = dict(lock["shared_PULSNAR_parameters"])
    shared["scar"] = True

    with tempfile.TemporaryDirectory(prefix="pulscar_repro_") as temp_dir:
        prior = Path.cwd()
        try:
            os.chdir(temp_dir)
            estimator = PULSNAR.PULSNARClassifier(classifier="xgboost", rseed=PULSCAR_SEED, **shared)
            result = estimator.pulsnar(
                discovery_x, labels, tru_label=None,
                rec_list=np.arange(len(training), dtype=np.int64),
            )
            prediction_file = Path(temp_dir) / Path(result["prediction_file"]).name
            predictions = pd.read_csv(prediction_file, sep="\t").sort_values("rec_id").reset_index(drop=True)
        finally:
            os.chdir(prior)

    required = {"rec_id", "ml_label", "calibrated_prob"}
    if not required.issubset(predictions.columns):
        raise ValueError(f"Official PULSCAR output is missing columns: {sorted(required-set(predictions.columns))}")
    if len(predictions) != len(training):
        raise ValueError("Official PULSCAR output row count changed")
    rec_id = pd.to_numeric(predictions["rec_id"], errors="raise").astype(int).to_numpy()
    if not np.array_equal(rec_id, np.arange(len(training), dtype=int)):
        raise ValueError("Official PULSCAR record order changed")
    returned_label_values = pd.to_numeric(predictions["ml_label"], errors="raise")
    if not returned_label_values.isin([0, 1]).all():
        raise ValueError("Official PULSCAR returned non-binary labels")
    returned_labels = returned_label_values.astype(int).to_numpy()
    if not np.array_equal(returned_labels, labels):
        raise ValueError("Official PULSCAR labels changed or became misaligned")
    alpha = float(result["estimated_alpha"])
    posterior = pd.to_numeric(predictions["calibrated_prob"], errors="raise").to_numpy(float)
    if not 0 <= alpha <= 1:
        raise ValueError("Official PULSCAR alpha is outside [0,1]")
    if not np.isfinite(posterior).all() or np.any((posterior < 0) | (posterior > 1)):
        raise ValueError("Official PULSCAR probabilities are invalid")
    unlabeled_pos = np.flatnonzero(labels == 0)
    n_select = int(math.floor(alpha * len(unlabeled_pos) + 0.5))
    ranking = pd.DataFrame({
        "position": unlabeled_pos,
        "posterior": posterior[unlabeled_pos],
        "patient": training.iloc[unlabeled_pos][ID_COL].astype(str).to_numpy(),
    })
    ranking["tie"] = ranking["patient"].map(lambda x: hashlib.sha256(f"full_pulscar|{x}".encode()).hexdigest())
    ranking = ranking.sort_values(["posterior", "tie"], ascending=[False, True], kind="stable")
    selected = np.zeros(len(training), dtype=bool)
    selected[ranking.iloc[:n_select].position.to_numpy(int)] = True
    if np.any(selected & (labels == 1)):
        raise AssertionError("PULSCAR selected an observed positive")
    return alpha, selected

if RUN_PULSCAR_DISCOVERY:
    alpha, probable_positive = run_official_pulscar(training, paths["software_lock"], paths["pulscar_wheel"])
    if not np.isclose(alpha, EXPECTED_ALPHA, atol=5e-5):
        raise AssertionError(f"Expected alpha≈{EXPECTED_ALPHA}; reproduced {alpha}")
    if int(probable_positive.sum()) != EXPECTED_TRAIN_SELECTED:
        raise AssertionError(f"Expected 265 probable positives; reproduced {probable_positive.sum()}")
    print({"alpha": alpha, "selected_probable_positive": int(probable_positive.sum())})
else:
    alpha = EXPECTED_ALPHA
    probable_positive = None
    print({"expected_alpha": alpha, "expected_selected_probable_positive": EXPECTED_TRAIN_SELECTED})

## 8. Final locked XGBoost refit and outcome-blind scoring

The locked hyperparameters were selected using five training folds and 40 Optuna trials after the training-only method screen. This reproduction cell uses those frozen parameters directly. Validation and internal-test predictors are transformed and scored **before** their separate outcome files are linked.

In [ ]:
if EXECUTION_MODE == "authorized_full_rerun":
    from xgboost import XGBClassifier
    import joblib

def make_locked_model():
    return XGBClassifier(
        **LOCKED_XGB_PARAMETERS,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=MODEL_SEED,
        verbosity=0,
    )

if training is not None:
    observed_train_y = pd.to_numeric(training[LABEL_COL], errors="raise").astype(int).to_numpy()
    final_train_y = observed_train_y.copy()
    final_train_y[probable_positive] = 1
    weights = pd.to_numeric(training[WEIGHT_COL], errors="raise").to_numpy(float)
    weights /= weights.mean()
    model = make_locked_model()
    model.fit(x_train, final_train_y, sample_weight=weights)
    validation_scores = model.predict_proba(transform(validation_x, preprocessor))[:, 1]
    test_scores = model.predict_proba(transform(test_x, preprocessor))[:, 1]
    # Scores are frozen here, before either outcome table is linked.
    pd.DataFrame({ID_COL: validation_x[ID_COL], "score": validation_scores}).to_parquet(OUTPUT_DIR / "validation_scores_outcome_blind.parquet", index=False)
    pd.DataFrame({ID_COL: test_x[ID_COL], "score": test_scores}).to_parquet(OUTPUT_DIR / "internal_test_scores_outcome_blind.parquet", index=False)
    joblib.dump(model, OUTPUT_DIR / "locked_model.joblib")
    joblib.dump(preprocessor, OUTPUT_DIR / "locked_preprocessor.joblib")
    print("Outcome-blind score files written; outcomes have not yet been linked.")
else:
    model = validation_scores = test_scores = None
    print("Specification check: model fitting skipped.")

## 9. Locked evaluation and stratified patient bootstrap

Average precision is primary. AUROC, Brier score, and log loss are secondary. Bootstrap resampling is stratified by observed label and performed at the patient level with 500 replicates and fixed seeds. Because the negative class is unlabeled rather than confirmed disease-free, all metrics are explicitly **observed-label metrics**.

In [ ]:
if EXECUTION_MODE == "authorized_full_rerun":
    from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score

def metric_values(labels, scores):
    labels = np.asarray(labels, dtype=int)
    scores = np.clip(np.asarray(scores, dtype=float), 1e-15, 1 - 1e-15)
    return {
        "Average_precision": float(average_precision_score(labels, scores)),
        "ROC_AUC": float(roc_auc_score(labels, scores)),
        "Brier_score": float(brier_score_loss(labels, scores)),
        "Log_loss": float(log_loss(labels, scores, labels=[0, 1])),
        "Observed_prevalence": float(labels.mean()),
        "Mean_score": float(scores.mean()),
    }

def stratified_patient_bootstrap(labels, scores, seed, replicates=500):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    positive = np.flatnonzero(labels == 1)
    unlabeled = np.flatnonzero(labels == 0)
    rng = np.random.default_rng(seed)
    rows = []
    for replicate in range(1, replicates + 1):
        sampled = np.concatenate([
            rng.choice(positive, len(positive), replace=True),
            rng.choice(unlabeled, len(unlabeled), replace=True),
        ])
        rows.append({"Replicate": replicate, **metric_values(labels[sampled], scores[sampled])})
    boot = pd.DataFrame(rows)
    point = metric_values(labels, scores)
    ci = pd.DataFrame([
        {"Measure": measure, "Point_estimate": point[measure],
         "Bootstrap_95CI_lower": boot[measure].quantile(0.025),
         "Bootstrap_95CI_upper": boot[measure].quantile(0.975)}
        for measure in ["Average_precision", "ROC_AUC", "Brier_score", "Log_loss"]
    ])
    return point, boot, ci

def link_outcomes(predictors, outcomes, scores, expected_positive):
    left = pd.DataFrame({"_key": patient_key(predictors[ID_COL]), "score": scores})
    right = outcomes[[ID_COL, LABEL_COL]].copy()
    right["_key"] = patient_key(right[ID_COL])
    merged = left.merge(right[["_key", LABEL_COL]], on="_key", how="left", validate="one_to_one")
    if merged[LABEL_COL].isna().any() or int(merged[LABEL_COL].sum()) != expected_positive:
        raise ValueError("Outcome linkage failed its frozen contract")
    return merged

if training is not None:
    validation_eval = link_outcomes(validation_x, validation_y, validation_scores, 1_089)
    test_eval = link_outcomes(test_x, test_y, test_scores, 1_071)
    val_point, val_boot, val_ci = stratified_patient_bootstrap(validation_eval[LABEL_COL], validation_eval.score, VALIDATION_BOOTSTRAP_SEED, BOOTSTRAP_REPLICATES)
    test_point, test_boot, test_ci = stratified_patient_bootstrap(test_eval[LABEL_COL], test_eval.score, TEST_BOOTSTRAP_SEED, BOOTSTRAP_REPLICATES)
    val_ci.assign(Cohort="Validation").to_csv(OUTPUT_DIR / "validation_metrics_bootstrap_ci.csv", index=False)
    test_ci.assign(Cohort="Internal test").to_csv(OUTPUT_DIR / "internal_test_metrics_bootstrap_ci.csv", index=False)
    display(val_ci.assign(Cohort="Validation"))
    display(test_ci.assign(Cohort="Internal test"))

## 10. Frozen-result verification

These are the manuscript values produced by the locked runs. A full rerun must agree within the declared numerical tolerances; bootstrap interval endpoints can vary slightly across compatible numerical libraries, but point estimates and cohort counts should reproduce closely.

In [ ]:
EXPECTED_RESULTS = pd.DataFrame([
    {"cohort": "Validation", "measure": "Average_precision", "estimate": 0.890622, "ci_lower": 0.875056, "ci_upper": 0.904042},
    {"cohort": "Validation", "measure": "ROC_AUC", "estimate": 0.975408, "ci_lower": 0.969404, "ci_upper": 0.980504},
    {"cohort": "Validation", "measure": "Brier_score", "estimate": 0.019340, "ci_lower": 0.017734, "ci_upper": 0.021231},
    {"cohort": "Validation", "measure": "Log_loss", "estimate": 0.077229, "ci_lower": 0.070622, "ci_upper": 0.084746},
    {"cohort": "Internal test", "measure": "Average_precision", "estimate": 0.881898, "ci_lower": 0.865177, "ci_upper": 0.896663},
    {"cohort": "Internal test", "measure": "ROC_AUC", "estimate": 0.968680, "ci_lower": 0.962542, "ci_upper": 0.974144},
    {"cohort": "Internal test", "measure": "Brier_score", "estimate": 0.018833, "ci_lower": 0.017103, "ci_upper": 0.020719},
    {"cohort": "Internal test", "measure": "Log_loss", "estimate": 0.079130, "ci_lower": 0.072410, "ci_upper": 0.086977},
])
display(EXPECTED_RESULTS)

if training is not None:
    reproduced = {
        ("Validation", k): v for k, v in val_point.items()
    } | {
        ("Internal test", k): v for k, v in test_point.items()
    }
    checks = []
    for row in EXPECTED_RESULTS.itertuples(index=False):
        observed = reproduced[(row.cohort, row.measure)]
        tolerance = 5e-4 if row.measure in {"Average_precision", "ROC_AUC"} else 1e-4
        checks.append({"cohort": row.cohort, "measure": row.measure, "expected": row.estimate,
                       "reproduced": observed, "absolute_difference": abs(observed-row.estimate),
                       "tolerance": tolerance, "passed": abs(observed-row.estimate) <= tolerance})
    checks = pd.DataFrame(checks)
    display(checks)
    if not checks.passed.all():
        raise AssertionError("At least one locked point estimate did not reproduce")

## 11. Post-hoc clinical comparator: eGFR alone

The comparator is descriptive and is not used to select or alter the model. Lower eGFR receives higher review priority; missing eGFR is assigned the lowest comparator priority. Differences in average precision are paired within the same bootstrap samples.

In [ ]:
def low_egfr_priority(values):
    values = pd.to_numeric(values, errors="coerce").to_numpy(float)
    score = -values
    finite = np.isfinite(score)
    floor = np.min(score[finite]) - 1.0 if finite.any() else -1.0
    score[~finite] = floor
    return score

EXPECTED_EGFR_COMPARATORS = pd.DataFrame([
    {"comparator": "Current eGFR alone", "AP": 0.777211, "AP_difference_vs_locked_model": -0.104687, "ci_lower": -0.125620, "ci_upper": -0.085867},
    {"comparator": "Minimum eGFR within 24 months alone", "AP": 0.784674, "AP_difference_vs_locked_model": -0.097224, "ci_lower": -0.117868, "ci_upper": -0.079180},
    {"comparator": "Mean eGFR within 24 months alone", "AP": 0.777907, "AP_difference_vs_locked_model": -0.103991, "ci_lower": -0.125191, "ci_upper": -0.085160},
])
display(EXPECTED_EGFR_COMPARATORS)

if training is not None:
    comparator_map = {
        "Current eGFR alone": "DERIVED_EGFR_CKD_EPI_2021",
        "Minimum eGFR within 24 months alone": "EGFR_MIN_24M",
        "Mean eGFR within 24 months alone": "EGFR_MEAN_24M",
    }
    y = test_eval[LABEL_COL].to_numpy(int)
    locked_score = test_eval["score"].to_numpy(float)
    comparator_scores = {
        name: low_egfr_priority(test_x[column])
        for name, column in comparator_map.items()
    }
    locked_ap = float(average_precision_score(y, locked_score))
    positive = np.flatnonzero(y == 1)
    unlabeled = np.flatnonzero(y == 0)
    rng = np.random.default_rng(EGFR_COMPARATOR_BOOTSTRAP_SEED)
    paired_rows = []
    for replicate in range(1, BOOTSTRAP_REPLICATES + 1):
        sampled = np.concatenate([
            rng.choice(positive, len(positive), replace=True),
            rng.choice(unlabeled, len(unlabeled), replace=True),
        ])
        sampled_model_ap = average_precision_score(y[sampled], locked_score[sampled])
        for name, score in comparator_scores.items():
            sampled_comparator_ap = average_precision_score(y[sampled], score[sampled])
            paired_rows.append({
                "replicate": replicate,
                "comparator": name,
                "comparator_AP_minus_locked_model_AP": float(sampled_comparator_ap-sampled_model_ap),
            })
    paired_bootstrap = pd.DataFrame(paired_rows)
    rows = []
    for name, column in comparator_map.items():
        score = comparator_scores[name]
        differences = paired_bootstrap.loc[
            paired_bootstrap["comparator"].eq(name),
            "comparator_AP_minus_locked_model_AP",
        ]
        rows.append({
            "comparator": name,
            "AP": float(average_precision_score(y, score)),
            "ROC_AUC": float(roc_auc_score(y, score)),
            "AP_difference_vs_locked_model": float(average_precision_score(y, score)-locked_ap),
            "ci_lower": float(differences.quantile(0.025)),
            "ci_upper": float(differences.quantile(0.975)),
            "bootstrap_replicates": BOOTSTRAP_REPLICATES,
        })
    egfr_results = pd.DataFrame(rows)
    egfr_results.to_csv(OUTPUT_DIR / "internal_test_egfr_comparator_paired_bootstrap.csv", index=False)
    display(egfr_results)

    comparison = egfr_results.merge(
        EXPECTED_EGFR_COMPARATORS,
        on="comparator",
        suffixes=("_reproduced", "_expected"),
        validate="one_to_one",
    )
    comparison["AP_passed"] = np.abs(comparison["AP_reproduced"]-comparison["AP_expected"]) <= 5e-4
    comparison["difference_passed"] = np.abs(
        comparison["AP_difference_vs_locked_model_reproduced"]-
        comparison["AP_difference_vs_locked_model_expected"]
    ) <= 5e-4
    comparison["CI_passed"] = (
        np.abs(comparison["ci_lower_reproduced"]-comparison["ci_lower_expected"]) <= 2e-3
    ) & (
        np.abs(comparison["ci_upper_reproduced"]-comparison["ci_upper_expected"]) <= 2e-3
    )
    display(comparison)
    if not comparison[["AP_passed", "difference_passed", "CI_passed"]].to_numpy().all():
        raise AssertionError("At least one paired eGFR comparator result did not reproduce")

## 12. Exact TreeSHAP and global importance

SHAP is computed only after the locked model has been evaluated. Contributions are on the raw model-margin scale. Mean absolute SHAP describes global influence, not causal importance. The three eGFR-derived variables remain separate model inputs in the global analysis; they were grouped only for the clinician-facing local display.

In [ ]:
EXPECTED_TOP_SHAP = pd.DataFrame([
    (1, "Minimum eGFR within the 24-month observation window", 2.334343),
    (2, "Pulse", 0.409563),
    (3, "Body mass index", 0.345273),
    (4, "Current derived eGFR", 0.336423),
    (5, "White blood cell count", 0.331259),
    (6, "Absolute lymphocyte count", 0.255227),
    (7, "Mean eGFR within the 24-month observation window", 0.253602),
    (8, "Sodium", 0.230628),
    (9, "Absolute eosinophil count", 0.230356),
    (10, "Systolic blood pressure", 0.227373),
    (11, "Age at index", 0.217751),
    (12, "Blood urea nitrogen", 0.207246),
    (13, "Hemoglobin", 0.196126),
    (14, "Potassium", 0.154526),
    (15, "Gender", 0.138132),
], columns=["rank", "feature", "mean_absolute_SHAP_raw_margin"])
display(EXPECTED_TOP_SHAP)

if RUN_SHAP:
    import shap
    x_test_matrix = transform(test_x, preprocessor)
    explainer = shap.TreeExplainer(model, model_output="raw")
    shap_values = np.asarray(explainer.shap_values(x_test_matrix), dtype=float)
    if shap_values.shape != x_test_matrix.shape:
        raise ValueError("Unexpected TreeSHAP shape")
    global_shap = pd.DataFrame({
        "transformed_feature": preprocessor["names"],
        "mean_absolute_SHAP_raw_margin": np.abs(shap_values).mean(axis=0),
    }).sort_values("mean_absolute_SHAP_raw_margin", ascending=False)
    global_shap.to_csv(OUTPUT_DIR / "global_treeshap_transformed_features.csv", index=False)
    display(global_shap.head(15))

## 13. Training-only sensitivity and baseline results

These tables reproduce prespecified supplementary analyses. The supervised algorithms are conventional predictive baselines and must not be described as direct substitutes for the PU discovery method.

In [ ]:
IMPUTATION_SENSITIVITY = pd.DataFrame([
    {"strategy": "Mean imputation", "OOF_AP": 0.885465, "delta_vs_mean": 0.0, "ci_lower": np.nan, "ci_upper": np.nan},
    {"strategy": "Median imputation", "OOF_AP": 0.882509, "delta_vs_mean": -0.002956, "ci_lower": -0.004369, "ci_upper": -0.001533},
    {"strategy": "Five-draw MICE", "OOF_AP": 0.873407, "delta_vs_mean": -0.012058, "ci_lower": -0.015018, "ci_upper": -0.009226},
])
BASELINE_BENCHMARK = pd.DataFrame([
    {"method": "Naive XGBoost", "OOF_AP": 0.884355},
    {"method": "Histogram gradient boosting", "OOF_AP": 0.881135},
    {"method": "PULSCAR + XGBoost", "OOF_AP": 0.881126},
    {"method": "Random forest", "OOF_AP": 0.867673},
    {"method": "Extra trees", "OOF_AP": 0.864969},
    {"method": "Penalized logistic regression", "OOF_AP": 0.815154},
])
display(IMPUTATION_SENSITIVITY)
display(BASELINE_BENCHMARK)

## 14. Export a machine-readable reproducibility manifest

The manifest records the execution mode, frozen design, expected outputs, software versions, and hashes of every available restricted input and reproduced output. It contains paths and hashes only—never patient-level values.

In [ ]:
available_input_hashes = {name: sha256_file(path) for name, path in paths.items() if path.is_file()}
reproduced_files = [p for p in OUTPUT_DIR.rglob("*") if p.is_file()]
manifest = {
    "analysis": "CKD hidden-risk review prioritization",
    "notebook_version": "1.0",
    "execution_mode": EXECUTION_MODE,
    "pipeline": "official PULSCAR discovery -> mean-imputed XGBoost",
    "score_interpretation": "review-priority ranking; not a calibrated CKD probability or diagnosis",
    "cohort_contract": COHORT_CONTRACT.to_dict(orient="records"),
    "retained_features": RETAINED_FEATURES,
    "excluded_features": EXCLUDED_FEATURES,
    "locked_xgboost_parameters": LOCKED_XGB_PARAMETERS,
    "expected_training_alpha": EXPECTED_ALPHA,
    "expected_training_probable_positives": EXPECTED_TRAIN_SELECTED,
    "bootstrap_replicates": BOOTSTRAP_REPLICATES,
    "bootstrap_seeds": {
        "validation": VALIDATION_BOOTSTRAP_SEED,
        "internal_test": TEST_BOOTSTRAP_SEED,
        "egfr_comparator": EGFR_COMPARATOR_BOOTSTRAP_SEED,
    },
    "input_sha256": available_input_hashes,
    "output_sha256": {p.name: sha256_file(p) for p in reproduced_files},
    "software": software.to_dict(orient="records"),
    "restricted_data_in_notebook": False,
}
manifest_path = OUTPUT_DIR / "reproducibility_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(manifest_path)

## 15. Interpretation and reporting guardrails

- The model score is a continuous ranking output on `[0,1]`; multiplying by 100 is a display transformation only. It is not a calibrated probability of CKD.
- Performance is evaluated against documented CKD versus unlabeled status, not confirmed disease absence.
- uACR and four sparse 90-day eGFR-history variables were excluded by the pre-specified training-only `>70%` missingness rule.
- Validation/test outcomes are linked only after outcome-blind score files are frozen.
- The eGFR-only comparison and SHAP analyses are post-hoc descriptive analyses and do not modify the locked model.
- The exploratory clinician review requires its own protocol, payload, and response-analysis notebook; clinician data are not generated or embedded here.

**Data availability statement template:** Deidentified patient-level EHR data are not publicly distributable because of institutional and privacy restrictions. The complete analysis code, frozen feature specification, model hyperparameters, software manifest, and aggregate expected outputs are supplied. Qualified investigators may request controlled access subject to institutional approval and a data-use agreement.